# M12 — locked CIFAR-100 test confirmation

This notebook performs the final source-locked confirmation for the controlled RanPAC random-ReLU frontend. Test extraction is impossible until the authorization cell has validated the train-only cache and the exact M6/M11/M11b artifacts. Run every cell in order on a fresh Colab T4 runtime.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_m12_cifar_features'
PERSIST_ROOT='/content/drive/MyDrive/SRQ_M12_LOCKED'
OUTPUT_DIR=PERSIST_ROOT+'/output'
AUTHORIZATION=PERSIST_ROOT+'/m12_authorization.json'
CONFIG='configs/srq_generalization_m12_locked_test_confirmation.json'
RUNNER='tools/srq_generalization_m12.py'
SOURCE_NAMES={
 'm6':'srq_generalization_m6_width_sweep_train_only.zip',
 'm11':'srq_generalization_m11_adaptive_precision_train_only.zip',
 'm11b':'srq_generalization_m11b_scale_refined_train_only.zip'}
SOURCE_SHA={
 'm6':'b2739b9da023ebd2eedb6fdfe01c394e94f252773e847533b35350021c3d239e',
 'm11':'65ce03df4da7041833014628b59aac1167f77bde2d64348b9a8a1e4fe09370a7',
 'm11b':'f33153c24716a7c660044cacc1e56cf040ee63d314fd13be2c7d47cf4895a9cf'}
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Fresh clone, dependencies, GPU, clean-tree, and canonical-LF source locks.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
Path(PERSIST_ROOT).mkdir(parents=True,exist_ok=True)
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU.'
def sha_raw(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
EXPECTED={
 'configs/srq_generalization_m12_locked_test_confirmation.json':'210494d0b1882b524fe29397e25218d05b0e555a93286270c6badabb2b7abd82',
 'tools/srq_generalization_m12.py':'39f27cead5fece9f5ed45238e7a50df814267642a72f4e3566400c7ee85715a4',
 'methods/analytic_ridge/adaptive_upper.py':'d34eae6072ea3e8736b6fb940c7a0691c845d8217569c7b6fe33f4b9e143c5dc',
 'methods/analytic_ridge/backends.py':'cb97a6b65991e41af5f52302bcbdeac5ded6dc95774055c64c6eecdbbfb50ad3',
 'methods/analytic_ridge/__init__.py':'59babe9c4881a0991a7f5825958cd0ef8c41f4d6b88b1b7cfc843b6e9ba99aed',
 'tools/srq_generalization_m5.py':'4d08e27a825fb59d300ee5909542bca4bd550a8558bb137159a176f353739a84',
 'tools/experiment_runner.py':'b2c953eea312a98ce4146757fb46a7dd0e4ebe20aa139da59464921b11f8310c',
 'models/backbone.py':'941e449dc6e66ca4018fb0d3ab3218d97ec97f498b557ed220c8332e75850a46',
 'utils/data_utils.py':'3cf85993e231b068ad5ae2f96be608b2e50e9c52f98fb2387fd3badfb44b6764',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda'}
for path,expected in EXPECTED.items(): assert sha_source(path)==expected,(path,sha_source(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
print('GPU:',torch.cuda.get_device_name(0))
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('M12 SOURCE LOCK: PASS')

In [ ]:
# Focused tests must pass before any artifact or dataset is opened.
subprocess.run([sys.executable,'-m','pytest','-q','-p','no:cacheprovider','tests/test_srq_generalization_m12.py','tests/test_srq_generalization_m11.py','tests/test_srq_generalization_m6.py'],check=True)
print('M12 PREFLIGHT TESTS: PASS')

In [ ]:
# Upload the exact immutable M6, M11, and M11b train-only artifacts.
from google.colab import files
uploaded=files.upload()
assert set(uploaded)==set(SOURCE_NAMES.values()),('Upload exactly these files',SOURCE_NAMES.values())
SOURCE_PATHS={}
for key,name in SOURCE_NAMES.items():
 path=Path('/content')/name
 path.write_bytes(uploaded[name])
 assert sha_raw(path)==SOURCE_SHA[key],(name,sha_raw(path),SOURCE_SHA[key])
 SOURCE_PATHS[key]=str(path)
print('ALL THREE TRAIN-ONLY SOURCE ARTIFACTS: PASS')

In [ ]:
# Download CIFAR-100 and the exact frozen ViT-B/16 checkpoint.
import kagglehub
from huggingface_hub import hf_hub_download
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha_raw(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
print('DATA/CHECKPOINT READY:',CIFAR_ROOT,CHECKPOINT_PATH)

In [ ]:
# Materialize TRAIN features only. No test image is opened in this cell.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
 command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m12','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
 subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
print('TRAIN-ONLY CACHE READY; TEST ABSENT')

In [ ]:
# Freeze the complete train-only identity BEFORE test materialization.
COMMON=[ '--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--source-m6-artifact',SOURCE_PATHS['m6'],'--source-m11-artifact',SOURCE_PATHS['m11'],'--source-m11b-artifact',SOURCE_PATHS['m11b'],'--authorization',AUTHORIZATION,'--require-clean-git']
subprocess.run([sys.executable,'-u',RUNNER,'authorize',*COMMON],check=True)
assert Path(AUTHORIZATION).is_file()
print('LOCKED AUTHORIZATION CREATED')

In [ ]:
# The runner now materializes the official TEST features under authorization.
subprocess.run([sys.executable,'-u',RUNNER,'extract-test',*COMMON,'--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)],check=True)
assert (Path(FEATURE_CACHE_DIR)/'test.pt').is_file()
print('AUTHORIZED TEST CACHE READY')

In [ ]:
# Run all 36 locked units. Existing identity-matched units resume safely.
print('M12 START: 6 paired replicates x 2 widths x 3 frozen methods.',flush=True)
completed=subprocess.run([sys.executable,'-u',RUNNER,'run',*COMMON,'--output-dir',OUTPUT_DIR,'--device','cuda'])
result_path=Path(OUTPUT_DIR)/'m12_results.json'
assert result_path.is_file(),'M12 stopped before producing a result; preserve full runner output.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('GATES:',json.dumps(result['gates'],indent=2))
assert completed.returncode==0 and result['status']=='COMPLETE_M12_LOCKED_TEST_CONFIRMATION','M12 integrity failed; do not retry based on accuracy.'

In [ ]:
# Human-readable locked summary. Accuracy is reported, never gated.
for width_result in result['aggregate']:
 print('\nWIDTH',width_result['width'])
 for method,metrics in width_result['methods'].items():
  aia=metrics['average_incremental_accuracy_percent']; final=metrics['final_accuracy_percent']
  print(f"{method:22s} AIA={aia['mean']:.4f}+/-{aia['sample_standard_deviation']:.4f} Final={final['mean']:.4f}+/-{final['sample_standard_deviation']:.4f} State={metrics['final_total_persistent_bytes']/2**20:.2f} MiB")

In [ ]:
# Export two paper-ready descriptive figures (no post-test selection).
import matplotlib.pyplot as plt
labels={'exact':'Exact','p2b_int8':'P2B INT8/FP32','adaptive_int8_fp16':'Adaptive INT8/FP16'}
colors={'exact':'#333333','p2b_int8':'#2878B5','adaptive_int8_fp16':'#D95319'}
fig,ax=plt.subplots(figsize=(6.2,4.2))
for wr in result['aggregate']:
 for method,metrics in wr['methods'].items():
  ax.scatter(metrics['final_total_persistent_bytes']/2**20,metrics['average_incremental_accuracy_percent']['mean'],s=55,color=colors[method])
  ax.annotate(f"{labels[method]} {wr['width']//1000}k",(metrics['final_total_persistent_bytes']/2**20,metrics['average_incremental_accuracy_percent']['mean']),xytext=(4,4),textcoords='offset points',fontsize=8)
ax.set_xlabel('Persistent state (MiB)');ax.set_ylabel('Test AIA (%)');ax.grid(alpha=.25);fig.tight_layout();fig.savefig(Path(OUTPUT_DIR)/'m12_accuracy_state.svg')
fig,axes=plt.subplots(1,2,figsize=(10,4),sharey=True)
for axis,width in zip(axes,[10000,20000]):
 for method in labels:
  selected=[u for u in result['units'] if u['identity']['width']==width and u['identity']['method']==method]
  means=[sum(u['records'][t]['test_accuracy_percent'] for u in selected)/len(selected) for t in range(10)]
  axis.plot(range(1,11),means,label=labels[method],color=colors[method])
 axis.set_title(f'Width {width:,}');axis.set_xlabel('Task');axis.grid(alpha=.25)
axes[0].set_ylabel('Seen-class test accuracy (%)');axes[1].legend(fontsize=8);fig.tight_layout();fig.savefig(Path(OUTPUT_DIR)/'m12_task_trajectory.svg')
plt.show()

In [ ]:
# Create the compact auditable artifact; caches and source ZIPs are excluded.
import zipfile
export=Path('/content/srq_generalization_m12_locked_test_confirmation.zip')
members=[Path(OUTPUT_DIR)/'m12_results.json',Path(OUTPUT_DIR)/'m12_summary.csv',Path(OUTPUT_DIR)/'m12_accuracy_state.svg',Path(OUTPUT_DIR)/'m12_task_trajectory.svg',Path(AUTHORIZATION),Path(CONFIG),Path('docs/research/SRQ_GENERALIZATION_M12_PROTOCOL.md')]
members.extend(sorted((Path(OUTPUT_DIR)/'units').glob('*.json')))
manifest={}
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as archive:
 for path in members:
  arcname=('units/'+path.name) if path.parent.name=='units' else path.name
  archive.write(path,arcname)
  manifest[arcname]=sha_raw(path)
 archive.writestr('MANIFEST.json',json.dumps({'schema_version':1,'artifact':'M12 locked test confirmation','files':manifest},indent=2)+'\n')
print('EXPORT:',export,'SHA-256:',sha_raw(export),'SIZE:',export.stat().st_size)
files.download(str(export))